### Setup and Initialization

In this section, we load environment variables (like your `GOOGLE_API_KEY`), ensure the project root is in the system path so we can import local utilities, and initialize the Gemini client. We also specify the model version we'll be using.

In [16]:
# Load env variables and create client
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Add project root to sys.path to import from utils
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
model = "gemini-2.5-flash-lite"

### Conversation Management Helpers

These helper functions simplify the process of building a conversation history. Gemini expects a specific structure for messages (`role` and `parts`). We also define a `chat` function that incorporates a retry mechanism to handle potential rate limits or transient errors.

In [17]:
# Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "parts": [{"text": text}]})


def add_model_message(messages, text):
    messages.append({"role": "model", "parts": [{"text": text}]})


def chat(messages, system_instruction=None, temperature=1.0, stop_sequences=None):
    config = types.GenerateContentConfig(
        temperature=temperature,
        stop_sequences=stop_sequences,
        system_instruction=system_instruction,
        max_output_tokens=1000,
    )
    
    # Using the shared retry helper for robustness
    response = generate_content_with_retry(
        client=client,
        model=model,
        contents=messages,
        config=config,
    )
    return response.text

### Defining Tools

Here we define the Python functions that the model can call. One of Gemini's strengths is the ability to automatically generate tool schemas from standard Python function signatures and docstrings. We group these functions into a `tools` list for easy configuration.

In [18]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str: str, duration: float = 0, unit: str = "days", input_format: str = "%Y-%m-%d"
) -> str:
    """
    Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format.
    
    Args:
        datetime_str: The input datetime string (e.g., '2025-04-03').
        duration: The amount of time to add. Can be negative for past dates. Defaults to 0.
        unit: The unit of time ('seconds', 'minutes', 'hours', 'days', 'weeks', 'months', 'years'). Defaults to 'days'.
        input_format: Python strptime format for parsing datetime_str. Defaults to '%Y-%m-%d'.
    """
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + int(duration)
        year = date.year + (month - 1) // 12
        month = (month - 1) % 12 + 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + int(duration))
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content: str, timestamp: str):
    """
    Creates a timed reminder that will notify the user at the specified time with the provided content.
    
    Args:
        content: The message text for the reminder.
        timestamp: ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) for when the reminder should trigger.
    """
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


# In the Gemini Python SDK, you can pass functions directly to the tools list.
# The SDK will automatically generate the schema from the function signature and docstring.
tools = [add_duration_to_datetime, set_reminder]

pass

### Implementing Tool Use

Finally, we implement a function that enables **Automatic Function Calling**. This allows Gemini to decide which tool to use, call it with the correct parameters, and use the output to fulfill the user's request, all within a single high-level interaction.

In [19]:
# Example of calling with tools in a multi-turn conversation
def chat_with_tools(messages, prompt, system_instruction=None):
    """
    Sends a message to the model with automatic tool calling enabled and updates history.
    """
    add_user_message(messages, prompt)
    
    # config with tools and automatic function calling enabled
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=tools,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False)
    )
    
    # Using retry helper
    response = generate_content_with_retry(
        client=client,
        model=model,
        contents=messages,
        config=config,
    )
    
    # Persist the model's response in the messages list to maintain context for next turns
    messages.append(response.candidates[0].content)
    
    return response

In [20]:
# Multi-turn conversation example
messages = []
system_prompt = "You are a helpful scheduling assistant. Use the provided tools to calculate times and set reminders."

print("--- Turn 1 ---")
prompt1 = "I have a meeting on 2025-04-10 12:00:00. Can you set a reminder for me 2 hours before that?"
response1 = chat_with_tools(messages, prompt1, system_instruction=system_prompt)
print(f"Assistant: {response1.text}")

print("\n--- Turn 2 ---")
prompt2 = "Actually, make it 3 hours before instead."
response2 = chat_with_tools(messages, prompt2, system_instruction=system_prompt)
print(f"Assistant: {response2.text}")

--- Turn 1 ---
----
Setting the following reminder for 2025-04-10T10:00:00:
Reminder for meeting
----
Assistant: None

--- Turn 2 ---
----
Setting the following reminder for 2025-04-10T09:00:00:
Reminder for meeting
----
Assistant: I have set a reminder for you for April 10, 2025 at 09:00:00 AM.
